# Beam Noise Diffusion (BND) Experiment

This notebook clones the latest code from GitHub into the Colab runtime. Google Drive is used only for persistent runtime artifacts: Hugging Face cache, generated images, result JSON files, and CSV summaries.


## Step 0: Check GPU

Verify which GPU Colab assigned before loading the diffusion model.


In [ ]:
!nvidia-smi


## Step 1: Mount Google Drive

Drive is used only as persistent storage for `cache/` and `outputs/`.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Step 2: Clone Code from GitHub

The repository is cloned into the temporary Colab runtime. This keeps executable code separate from persistent outputs.


In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/nguyenphamngocquy/beam-noise-diffusion.git"
CODE_ROOT = Path("/content/beam-noise-diffusion")
STORAGE_ROOT = Path("/content/drive/MyDrive/beam-noise-diffusion-storage")

if CODE_ROOT.exists():
    shutil.rmtree(CODE_ROOT)

subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(CODE_ROOT)], check=True)
sys.path.insert(0, str(CODE_ROOT))

print("Code root:", CODE_ROOT)
print("Storage root:", STORAGE_ROOT)


## Step 3: Install Dependencies

Install dependencies from the cloned repository's `requirements.txt`.


In [ ]:
import subprocess
import sys

subprocess.run([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "-r",
    str(CODE_ROOT / "requirements.txt"),
], check=True)


## Step 4: Load Config and Prompts

Configs and prompts are read from the cloned repository. Runtime outputs are written to `STORAGE_ROOT` on Drive.


In [ ]:
from bnd.config import load_config, load_prompts
from bnd.io import ensure_experiment_dirs

CONFIG_NAME = "bnd_light"  # Change to "bnd_full" for A100/full run
ensure_experiment_dirs(STORAGE_ROOT)
CFG = load_config(CODE_ROOT, CONFIG_NAME)
PROMPTS = load_prompts(CODE_ROOT)
RUN_NAME = CFG["name"]

print("Config:", RUN_NAME)
print("Perturbation:", CFG.get("perturbation_mode", "additive"))
print("Number of prompts:", len(PROMPTS))
print(PROMPTS[0])


## Step 5: Load Models

Stable Diffusion and CLIP are loaded from code in the cloned repo. Model files are cached on Drive under `STORAGE_ROOT/cache/hf_cache`.


In [ ]:
from bnd.sampling import get_gpu_name, load_sd_pipeline
from bnd.scoring import ClipScorer

print("GPU:", get_gpu_name())
CACHE_DIR = STORAGE_ROOT / "cache" / "hf_cache"
pipe = load_sd_pipeline(CFG, CACHE_DIR)
scorer = ClipScorer(CACHE_DIR)


## Step 6: Run Experiments

For each prompt, the notebook runs three methods: random baseline, Monte-Carlo selection, and BND. Set `PROMPT_LIMIT` to a small number for a smoke test.


In [ ]:
from bnd.algorithm import run_bnd, run_monte_carlo, run_random_baseline

PROMPT_LIMIT = None  # Set to a small number, e.g. 2, for a quick smoke test
items = PROMPTS if PROMPT_LIMIT is None else PROMPTS[:PROMPT_LIMIT]

for item in items:
    run_random_baseline(item, CFG, pipe, scorer, STORAGE_ROOT, RUN_NAME)
    run_monte_carlo(item, CFG, pipe, scorer, STORAGE_ROOT, RUN_NAME)
    run_bnd(item, CFG, pipe, scorer, STORAGE_ROOT, RUN_NAME)


## Step 7: Summarize Results

Collect JSON result files from Drive into CSV summaries and display aggregate metrics.


In [ ]:
from bnd.io import summarize_results

df, summary, group_summary = summarize_results(STORAGE_ROOT, RUN_NAME)
display(df.head())
display(summary)
display(group_summary)
